# GetLDmatrices — LD matrices for GWAS coloc loci

Cleaned from `LD_and_coloc.ipynb`. Writes
`code/results/coloc/LD/{gwas_loci}.tsv.gz`, the r matrices consumed by
**Fig. 4f** (and by the supplementary LocusCompare panels).

### What changed

The original had one hard-coded `getLDMatrix("chr17:43615230-44613099")` call
per locus, with the coordinates typed in by hand — four loci, four calls, no
loop. Those hand-typed regions turned out to be the **lead-SNP ±500 kb window
trimmed to the span of variants that actually exist inside it**, which is what
you get from eyeballing the VCF interactively.

This notebook takes the window straight from
`code/resources/gwas/LeadSnpWindows.bed`
(`rules/PrepareGWAS.smk :: ConcatGwasLeadSnpWindows`), which is exactly
lead ± 500,000 — verified for all four original loci:

| Locus | Bed window | = lead ± 500 kb |
|---|---|---|
| `chr12_57713053_N_N_IMSGC2019` | 57213053–58213053 | ✓ |
| `chr11_803017_N_N_GCST004988` | 303017–1303017 | ✓ |
| `chr11_578759_N_N_Hypothyroidism` | 78759–1078759 | ✓ |
| `chr17_44114525_N_N_bipolar_disorder` | 43614525–44614525 | ✓ |

**Does using the full ±500 kb instead of the trimmed region change anything?**
Barely, and not in a way that matters. Counting variants at AF ≥ 0.05 in the
flanks that the hand-typed regions cut off:

| Locus | Extra variants |
|---|---|
| `chr12_..._IMSGC2019` | 0 |
| `chr11_..._GCST004988` | 0 |
| `chr17_..._bipolar_disorder` | 1 |
| `chr11_..._Hypothyroidism` | 3 |

The single extra variant at the chr17 locus (the one Fig. 4f uses) is
`chr17_44613919_TG_T_b38` at **AF = 0.9994** — effectively monomorphic, and
`getLDMatrix` already drops zero-variance rows before correlating. So the
matrix is the same one Fig. 4f reads, give or take a column that carries no
information. Using the bed window is the better trade: reproducible for any
locus, instead of four sets of coordinates that can only be recovered by
squinting at a VCF.

In [1]:
suppressMessages({
    library(vcfR)
    library(genetics)
    library(tidyverse)
    library(data.table)
    library(BEDMatrix)
    library(glue)
    library(bedr)
})

BASE <- "/project/yangili1/cfbuenabadn/leafcutter2_paper"

VCF_FILE <- paste0("/project/yangili1/cdai/genome_index/hs38/GTEx_v7/",
                   "GTEx_Analysis_2017-06-05_v8_WGS_VCF_files_",
                   "GTEx_Analysis_2017-06-05_v8_WholeGenomeSeq_838Indiv_",
                   "Analysis_Freeze.SHAPEIT2_phased.vcf.gz")

LEAD_SNP_WINDOWS <- file.path(BASE, "code/resources/gwas/LeadSnpWindows.bed")
LD_DIR           <- file.path(BASE, "code/results/coloc/LD")

MIN_AF       <- 0.05      # minor allele frequency cutoff
MAX_MISSING  <- 838 / 20  # at most 5% of the 838 GTEx donors may be './.'

dir.create(LD_DIR, showWarnings = FALSE, recursive = TRUE)
cat("VCF        :", VCF_FILE, "\n")
cat("windows    :", LEAD_SNP_WINDOWS, "\n")
cat("writing to :", LD_DIR, "\n")

ERROR: Error in library(genetics): there is no package called ‘genetics’


In [ ]:
getLDMatrix <- function(region) {
    # Pairwise genotype correlation (r, not r^2) across GTEx donors for every
    # common variant in `region`. Kept as in LD_and_coloc.ipynb:
    #   * keep FORMAT == 'GT' records with AF >= MIN_AF
    #   * drop variants missing in more than MAX_MISSING donors
    #   * code 0|0 -> 0, het -> 1, 1|1 -> 2, .|. -> NA
    #   * impute NA to the per-donor mean, drop zero-variance variants
    # Consumers square it: Figure4_helpers does LD.astype(float)**2.
    gt_region <- bedr::tabix(region = region, file = VCF_FILE) %>%
        filter(FORMAT == 'GT')

    rownames(gt_region) <- gt_region$ID

    gt_filtered <- gt_region %>%
        mutate(AF = as.numeric(str_extract(INFO, "(?<=AF=)[0-9.]+"))) %>%
        filter(AF >= MIN_AF)

    gt_region <- gt_filtered[, grepl("^GTEX-", colnames(gt_filtered))]
    gt_region <- gt_region[((gt_region == '.|.') %>% rowSums()) <= MAX_MISSING, ]

    gt_region[gt_region == "0|0"] <- 0
    gt_region[(gt_region == "1|0") | (gt_region == "0|1")] <- 1
    gt_region[gt_region == "1|1"] <- 2
    gt_region[gt_region == ".|."] <- NA

    matrix_data_int <- gt_region %>% mutate_all(as.integer)
    imputed_data <- apply(matrix_data_int, 2,
                          function(x) ifelse(is.na(x), mean(x, na.rm = TRUE), x))

    row_variances <- apply(imputed_data, 1, var)
    filtered_matrix <- imputed_data[row_variances != 0, , drop = FALSE]

    filtered_matrix %>% t() %>% cor()
}


region_for_locus <- function(locus, windows) {
    # 'chr17:43614525-44614525' for a gwas_loci id, straight from the bed file.
    w <- windows %>% filter(gwas_loci == locus)
    if (nrow(w) != 1) stop(glue("expected 1 window for {locus}, found {nrow(w)}"))
    glue("{w$chrom[1]}:{w$start[1]}-{w$end[1]}")
}


write_ld_for_locus <- function(locus, windows, ld_dir = LD_DIR, overwrite = FALSE) {
    out <- file.path(ld_dir, paste0(locus, ".tsv.gz"))
    if (file.exists(out) && !overwrite) {
        cat("skip (exists):", basename(out), "\n"); return(invisible(out))
    }
    region <- region_for_locus(locus, windows)
    cat("building", locus, "over", region, "... ")
    LD <- getLDMatrix(region)
    LD %>% as.data.frame() %>% write_tsv(out)
    cat(nrow(LD), "x", ncol(LD), "variants ->", basename(out), "\n")
    invisible(out)
}

In [ ]:
# readr in this R build predates show_col_types; suppress the spec message instead
windows <- suppressMessages(
    read_tsv(LEAD_SNP_WINDOWS, col_names = c("chrom", "start", "end", "gwas_loci")))
cat(nrow(windows), "GWAS lead-SNP windows\n")

# Sanity check: every window is exactly lead +/- 500 kb, where the lead
# position is embedded in the locus id (chrN_POS_N_N_TRAIT).
windows <- windows %>%
    mutate(lead_pos = as.numeric(str_split_fixed(gwas_loci, "_", 3)[, 2]),
           half_width_ok = (lead_pos - start == 5e5) & (end - lead_pos == 5e5))
cat("windows that are exactly lead +/- 500kb:",
    sum(windows$half_width_ok), "/", nrow(windows), "\n")

In [ ]:
# The four loci LD_and_coloc.ipynb built by hand. Fig. 4f uses the bipolar one;
# the others back supplementary LocusCompare panels.
LOCI <- c(
    "chr17_44114525_N_N_bipolar_disorder",   # Fig. 4f  (ASB16)
    "chr12_57713053_N_N_IMSGC2019",
    "chr11_803017_N_N_GCST004988",
    "chr11_578759_N_N_Hypothyroidism"
)

# Set overwrite = TRUE to rebuild files that already exist. Each locus is a
# ~1 Mb window x 838 donors, so this is slow and memory-hungry.
for (locus in LOCI) write_ld_for_locus(locus, windows, overwrite = FALSE)

### Building LD for any other locus

`windows` carries all GWAS lead-SNP windows, so nothing here is specific to the
four above:

```r
write_ld_for_locus("chr11_64340263_N_N_Hypothyroidism", windows)
```

or, to build every locus a trait contributes:

```r
loci <- windows %>% filter(str_detect(gwas_loci, "bipolar_disorder")) %>% pull(gwas_loci)
for (locus in loci) write_ld_for_locus(locus, windows)
```

Be aware this is the expensive step in the coloc figures — one ~1 Mb window
across 838 donors per locus. If it ever needs to run at scale it belongs in a
Snakemake rule keyed on `{gwas_loci}`, since the region is already derivable
from `LeadSnpWindows.bed`.